# Deney 1 — Mouth ROI Swin V2 Tiny Baseline v3 — Resume Ready

Bu sürüm, önceki notebook'un standarda göre tespit edilen eksiklerini kapatır.

### v3 ile doğrulanan / güçlendirilenler
- Gerçek `new / resume` çalışma modu.
- `last.ckpt` üzerinden model + optimizer + scheduler + scaler + RNG + DataLoader generator state geri yükleme.
- Checkpoint continuity testi: atomik save/load sonrası aynı deterministik batch üzerinde aynı logits/loss doğrulaması.
- Colab uyumlu **modüler `src/deepfake_roi/`** yapısı.
- Runtime bağımlılıklarının `requirements_runtime.lock.txt` olarak kilitlenmesi.
- Kaynak modüllerin hash manifesti ve run klasörüne source snapshot.
- SSOT metadata schema audit: mevcut dataset'te olmayan kolonlar uydurulmaz, eksikler açıkça raporlanır.
- Pretrained Swin için ImageNet normalization kullanımının gerekçesi config ve audit içinde kayıt altına alınır.
- Video-level leakage gate, smoke test, NaN/Inf gate, fresh-inference gate korunur.
- Test threshold'u yalnızca validation setinden seçilir.
- Sonuçlar: `AISC Çalışmalar / Deneyler / Dilara / Deney 1 / Sonuçlar / <run_id>/`
- Varsayılan eğitim FP32: Transformer attention katmanlarında FP16 overflow riski ortadan kaldırıldı.
- AMP açılırsa GradScaler overflow'u kontrollü yönetir; geçici overflow fatal sayılmaz, ardışık overflow limiti vardır.
- Kritik hücrelerde eksik `copy/math/json/numpy/pandas` import bağımlılıkları giderildi.
- `best_epoch` ve early-stopping counter checkpoint'e alınır; resume davranışı gerçekten süreklidir.

In [1]:
import torch

print("CUDA kullanılabilir mi?:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU bulunamadı, CPU kullanılıyor.")

CUDA kullanılabilir mi?: True
GPU: Tesla T4


In [2]:
# ============================================================
# 0. OPTIONAL PINNED AUXILIARY DEPENDENCIES
# ============================================================
# Colab torch/torchvision ikilisini CUDA uyumluluğunu bozmamak için değiştirmiyoruz.
# Bu notebook unpinned pip install kullanmaz.
#
# Temiz bir ortamda yardımcı paketleri sabitlemek isterseniz:
INSTALL_PINNED_AUX = False

PINNED_AUX = [
    "pandas==2.2.3",
    "numpy==2.3.5",
    "scikit-learn==1.8.0",
    "matplotlib==3.10.8",
    "Pillow==12.2.0",
    "PyYAML==6.0.3",
    "tqdm==4.67.3",
]

if INSTALL_PINNED_AUX:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *PINNED_AUX
    ])
else:
    print("Pinned auxiliary install skipped; current Colab runtime will be audited and locked.")

Pinned auxiliary install skipped; current Colab runtime will be audited and locked.


In [3]:
# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# ============================================================
# 2. CREATE MODULAR RUNTIME PACKAGE STRUCTURE
# ============================================================
from pathlib import Path
import shutil

WORKSPACE = Path("/content/mouth_swinv2_experiment")
SRC_ROOT = WORKSPACE / "src"

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)

for p in [
    SRC_ROOT / "deepfake_roi",
    SRC_ROOT / "deepfake_roi" / "data",
    SRC_ROOT / "deepfake_roi" / "models",
    SRC_ROOT / "deepfake_roi" / "training",
    SRC_ROOT / "deepfake_roi" / "evaluation",
    SRC_ROOT / "deepfake_roi" / "utils",
]:
    p.mkdir(parents=True, exist_ok=True)

for init_file in [
    SRC_ROOT / "deepfake_roi" / "__init__.py",
    SRC_ROOT / "deepfake_roi" / "data" / "__init__.py",
    SRC_ROOT / "deepfake_roi" / "models" / "__init__.py",
    SRC_ROOT / "deepfake_roi" / "training" / "__init__.py",
    SRC_ROOT / "deepfake_roi" / "evaluation" / "__init__.py",
    SRC_ROOT / "deepfake_roi" / "utils" / "__init__.py",
]:
    init_file.write_text("", encoding="utf-8")

print("Workspace:", WORKSPACE)

Workspace: /content/mouth_swinv2_experiment


In [5]:
%%writefile /content/mouth_swinv2_experiment/src/deepfake_roi/utils/repro.py
from __future__ import annotations

import os
import random
from typing import Any, Dict

import numpy as np
import torch


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id: int) -> None:
    del worker_id
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def capture_rng_state() -> Dict[str, Any]:
    return {
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
    }


def restore_rng_state(state: Dict[str, Any]) -> None:
    random.setstate(state["python_rng_state"])
    np.random.set_state(state["numpy_rng_state"])
    torch.set_rng_state(state["torch_rng_state"])

    if torch.cuda.is_available() and state.get("cuda_rng_state") is not None:
        torch.cuda.set_rng_state_all(state["cuda_rng_state"])

Writing /content/mouth_swinv2_experiment/src/deepfake_roi/utils/repro.py


In [6]:
%%writefile /content/mouth_swinv2_experiment/src/deepfake_roi/utils/io.py
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path
from typing import Any, Dict, Iterable

import torch


CHECKPOINT_REQUIRED_KEYS = {
    "epoch",
    "model_state_dict",
    "optimizer_state_dict",
    "scheduler_state_dict",
    "scaler_state_dict",
    "best_metric_score",
    "best_threshold",
    "best_epoch",
    "epochs_without_improvement",
    "history",
    "config",
    "rng_state",
    "loader_generator_state",
}


def atomic_save_checkpoint(state: Dict[str, Any], target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    temp_path = target.with_suffix(target.suffix + ".tmp")

    if temp_path.exists():
        temp_path.unlink()

    try:
        torch.save(state, temp_path)

        loaded = torch.load(
            temp_path,
            map_location="cpu",
            weights_only=False,
        )

        missing = CHECKPOINT_REQUIRED_KEYS.difference(loaded.keys())
        if missing:
            raise RuntimeError(
                f"Checkpoint integrity failed. Missing keys: {sorted(missing)}"
            )

        if int(loaded["epoch"]) != int(state["epoch"]):
            raise RuntimeError("Checkpoint epoch integrity mismatch.")

        os.replace(temp_path, target)

    except Exception:
        if temp_path.exists():
            temp_path.unlink()
        raise


def load_checkpoint(path: Path, device: torch.device) -> Dict[str, Any]:
    if not path.is_file():
        raise FileNotFoundError(path)

    state = torch.load(
        path,
        map_location=device,
        weights_only=False,
    )

    missing = CHECKPOINT_REQUIRED_KEYS.difference(state.keys())
    if missing:
        raise RuntimeError(
            f"Checkpoint incomplete. Missing keys: {sorted(missing)}"
        )

    return state


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def save_json_atomic(data: Dict[str, Any], target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    os.replace(tmp, target)


def hash_source_tree(paths: Iterable[Path]) -> Dict[str, str]:
    return {
        str(path): sha256_file(path)
        for path in sorted(paths)
        if path.is_file()
    }

Writing /content/mouth_swinv2_experiment/src/deepfake_roi/utils/io.py


In [7]:
%%writefile /content/mouth_swinv2_experiment/src/deepfake_roi/data/dataset.py
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.models import Swin_V2_T_Weights


LABEL_TO_INDEX = {
    "real": 0.0,
    "fake": 1.0,
}


def build_transforms(
    image_size: int,
    augmentation: Dict[str, float],
):
    # Methodological choice:
    # For ImageNet-pretrained Swin V2, the normalization attached to the
    # pretrained weights is used. No validation/test statistics are learned.
    weights = Swin_V2_T_Weights.IMAGENET1K_V1
    weight_transform = weights.transforms()
    mean = weight_transform.mean
    std = weight_transform.std

    train_transform = transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(
                p=float(augmentation["horizontal_flip_probability"])
            ),
            transforms.RandomRotation(
                degrees=float(augmentation["rotation_degrees"])
            ),
            transforms.ColorJitter(
                brightness=float(augmentation["brightness"]),
                contrast=float(augmentation["contrast"]),
                saturation=float(augmentation["saturation"]),
                hue=float(augmentation["hue"]),
            ),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ]
    )

    eval_transform = transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ]
    )

    normalization_info = {
        "source": "Swin_V2_T_Weights.IMAGENET1K_V1",
        "mean": list(mean),
        "std": list(std),
        "learned_from_project_data": False,
        "uses_validation_or_test_statistics": False,
    }

    return train_transform, eval_transform, normalization_info


class MouthROIDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        transform,
        sample_col: str,
        video_col: str,
        label_col: str,
        resolved_path_col: str = "_resolved_image_path",
    ) -> None:
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform
        self.sample_col = sample_col
        self.video_col = video_col
        self.label_col = label_col
        self.resolved_path_col = resolved_path_col

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        row = self.df.iloc[index]
        path = Path(str(row[self.resolved_path_col]))

        if not path.is_file():
            raise FileNotFoundError(
                f"ROI image disappeared after audit: {path}"
            )

        try:
            with Image.open(path) as img:
                image = img.convert("RGB")
        except Exception as exc:
            raise RuntimeError(f"Image decode failed: {path}") from exc

        image = self.transform(image)

        label_text = str(row[self.label_col])
        if label_text not in LABEL_TO_INDEX:
            raise ValueError(f"Unexpected label: {label_text}")

        return {
            "image": image,
            "label": torch.tensor(
                LABEL_TO_INDEX[label_text],
                dtype=torch.float32,
            ),
            "sample_id": str(row[self.sample_col]),
            "video_id": str(row[self.video_col]),
            "path": str(path),
        }

Writing /content/mouth_swinv2_experiment/src/deepfake_roi/data/dataset.py


In [8]:
%%writefile /content/mouth_swinv2_experiment/src/deepfake_roi/models/swin.py
from __future__ import annotations

import torch
import torch.nn as nn
from torchvision.models import Swin_V2_T_Weights, swin_v2_t


class SwinV2TinyBinaryClassifier(nn.Module):
    def __init__(
        self,
        pretrained: bool = True,
        dropout: float = 0.20,
    ) -> None:
        super().__init__()

        weights = (
            Swin_V2_T_Weights.IMAGENET1K_V1
            if pretrained
            else None
        )

        self.backbone = swin_v2_t(weights=weights)

        in_features = self.backbone.head.in_features
        self.backbone.head = nn.Identity()

        self.classifier = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.classifier(features).squeeze(1)

Writing /content/mouth_swinv2_experiment/src/deepfake_roi/models/swin.py


In [9]:
%%writefile /content/mouth_swinv2_experiment/src/deepfake_roi/training/engine.py
from __future__ import annotations

from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def safe_roc_auc(y_true: np.ndarray, probs: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, probs))


def safe_pr_auc(y_true: np.ndarray, probs: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(average_precision_score(y_true, probs))


def compute_metrics(
    y_true: np.ndarray,
    probs: np.ndarray,
    threshold: float,
) -> Dict[str, float]:
    preds = (probs >= threshold).astype(np.int64)
    return {
        "accuracy": float(accuracy_score(y_true, preds)),
        "precision": float(
            precision_score(y_true, preds, zero_division=0)
        ),
        "recall": float(
            recall_score(y_true, preds, zero_division=0)
        ),
        "f1": float(
            f1_score(y_true, preds, zero_division=0)
        ),
        "roc_auc": safe_roc_auc(y_true, probs),
        "pr_auc": safe_pr_auc(y_true, probs),
    }


def select_best_f1_threshold(
    y_true: np.ndarray,
    probs: np.ndarray,
) -> Tuple[float, float]:
    if len(y_true) == 0:
        raise ValueError("Cannot select threshold from an empty validation set.")

    thresholds = np.linspace(0.05, 0.95, 181)
    best_threshold = 0.5
    best_f1 = -1.0

    for threshold in thresholds:
        preds = (probs >= threshold).astype(np.int64)
        score = f1_score(
            y_true,
            preds,
            zero_division=0,
        )
        if score > best_f1:
            best_f1 = float(score)
            best_threshold = float(threshold)

    return best_threshold, best_f1


def assert_finite_tensor(
    tensor: torch.Tensor,
    name: str,
) -> None:
    if not torch.isfinite(tensor).all():
        raise FloatingPointError(
            f"NaN/Inf detected in {name}."
        )


def assert_finite_gradients(model: nn.Module) -> None:
    for name, parameter in model.named_parameters():
        if parameter.grad is None:
            continue
        if not torch.isfinite(parameter.grad).all():
            raise FloatingPointError(
                f"NaN/Inf gradient in parameter: {name}"
            )


def run_epoch(
    *,
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    amp_enabled: bool,
    scaler,
    gradient_clip_norm: float,
    amp_overflow_abort_after: int = 8,
    optimizer: Optional[torch.optim.Optimizer] = None,
) -> Dict[str, Any]:
    training = optimizer is not None
    model.train(training)

    running_loss = 0.0
    total_samples = 0
    amp_overflow_steps = 0
    consecutive_amp_overflows = 0

    labels_all: List[float] = []
    probs_all: List[float] = []
    sample_ids: List[str] = []
    video_ids: List[str] = []
    paths: List[str] = []

    grad_context = (
        torch.enable_grad()
        if training
        else torch.no_grad()
    )

    with grad_context:
        progress = tqdm(
            loader,
            leave=False,
            desc="Train" if training else "Evaluate",
        )

        for batch in progress:
            images = batch["image"].to(
                device,
                non_blocking=True,
            )
            labels = batch["label"].to(
                device,
                non_blocking=True,
            )

            assert_finite_tensor(images, "input images")
            assert_finite_tensor(labels, "labels")

            if training:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=device.type,
                dtype=(
                    torch.float16
                    if device.type == "cuda"
                    else torch.bfloat16
                ),
                enabled=amp_enabled,
            ):
                logits = model(images)
                assert_finite_tensor(logits, "logits")
                loss = criterion(logits, labels)
                assert_finite_tensor(loss, "loss")

            if training:
                if amp_enabled:
                    # Dynamic loss scaling may transiently create Inf gradients.
                    # GradScaler is designed to skip that optimizer step and
                    # reduce its scale, so a single AMP overflow is not fatal.
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)

                    if gradient_clip_norm > 0:
                        torch.nn.utils.clip_grad_norm_(
                            model.parameters(),
                            max_norm=gradient_clip_norm,
                            error_if_nonfinite=False,
                        )

                    previous_scale = float(scaler.get_scale())
                    scaler.step(optimizer)
                    scaler.update()
                    current_scale = float(scaler.get_scale())

                    overflow = current_scale < previous_scale
                    if overflow:
                        amp_overflow_steps += 1
                        consecutive_amp_overflows += 1
                        optimizer.zero_grad(set_to_none=True)

                        if consecutive_amp_overflows >= amp_overflow_abort_after:
                            raise FloatingPointError(
                                "Persistent AMP gradient overflow: "
                                f"{consecutive_amp_overflows} consecutive steps. "
                                "Disable mixed precision or inspect inputs/model."
                            )
                    else:
                        consecutive_amp_overflows = 0

                else:
                    # FP32 baseline: non-finite gradients are always fatal.
                    loss.backward()
                    assert_finite_gradients(model)

                    if gradient_clip_norm > 0:
                        torch.nn.utils.clip_grad_norm_(
                            model.parameters(),
                            max_norm=gradient_clip_norm,
                            error_if_nonfinite=True,
                        )

                    optimizer.step()

            probs = (
                torch.sigmoid(logits)
                .detach()
                .float()
                .cpu()
                .numpy()
            )
            labels_np = (
                labels.detach()
                .float()
                .cpu()
                .numpy()
            )

            batch_size = int(images.shape[0])
            running_loss += float(loss.item()) * batch_size
            total_samples += batch_size

            labels_all.extend(labels_np.tolist())
            probs_all.extend(probs.tolist())
            sample_ids.extend(list(batch["sample_id"]))
            video_ids.extend(list(batch["video_id"]))
            paths.extend(list(batch["path"]))

            postfix = {"loss": f"{loss.item():.4f}"}
            if amp_enabled:
                postfix["amp_overflows"] = amp_overflow_steps
            progress.set_postfix(**postfix)

    if total_samples == 0:
        raise RuntimeError("No samples processed.")

    return {
        "loss": running_loss / total_samples,
        "labels": np.asarray(labels_all, dtype=np.int64),
        "probabilities": np.asarray(probs_all, dtype=np.float64),
        "sample_ids": sample_ids,
        "video_ids": video_ids,
        "paths": paths,
        "amp_overflow_steps": int(amp_overflow_steps),
    }

Writing /content/mouth_swinv2_experiment/src/deepfake_roi/training/engine.py


In [10]:
%%writefile /content/mouth_swinv2_experiment/src/deepfake_roi/evaluation/plots.py
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)


def save_figure(
    fig,
    output_path: Path,
    min_short_edge: int = 600,
) -> None:
    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight",
    )
    plt.close(fig)

    with Image.open(output_path) as image:
        if min(image.size) < min_short_edge:
            raise RuntimeError(
                f"Figure resolution failed: "
                f"{output_path} -> {image.size}"
            )


def generate_all_figures(
    *,
    history_df,
    labels,
    probabilities,
    threshold: float,
    figures_dir: Path,
    min_short_edge: int,
) -> None:
    figures_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    ax.plot(
        history_df["epoch"],
        history_df["train_loss"],
        label="Training Loss",
        linewidth=2,
    )
    ax.plot(
        history_df["epoch"],
        history_df["val_loss"],
        label="Validation Loss",
        linewidth=2,
        linestyle="--",
    )
    ax.set_title(
        "Training and Validation Loss Curve",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Epoch", fontsize=11)
    ax.set_ylabel("Loss", fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)
    save_figure(
        fig,
        figures_dir / "training_validation_loss.png",
        min_short_edge,
    )

    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    ax.plot(
        history_df["epoch"],
        history_df["train_accuracy"],
        label="Training Accuracy",
        linewidth=2,
    )
    ax.plot(
        history_df["epoch"],
        history_df["val_accuracy"],
        label="Validation Accuracy",
        linewidth=2,
        linestyle="--",
    )
    ax.set_title(
        "Training and Validation Accuracy",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Epoch", fontsize=11)
    ax.set_ylabel("Accuracy", fontsize=11)
    ax.set_ylim(0, 1.01)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)
    save_figure(
        fig,
        figures_dir / "training_validation_accuracy.png",
        min_short_edge,
    )

    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    ax.plot(
        history_df["epoch"],
        history_df["val_roc_auc"],
        label="Validation ROC-AUC",
        linewidth=2,
    )
    ax.set_title(
        "Validation ROC-AUC by Epoch",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Epoch", fontsize=11)
    ax.set_ylabel("ROC-AUC", fontsize=11)
    ax.set_ylim(0, 1.01)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)
    save_figure(
        fig,
        figures_dir / "validation_roc_auc.png",
        min_short_edge,
    )

    preds = (probabilities >= threshold).astype(np.int64)
    cm = confusion_matrix(
        labels,
        preds,
        labels=[0, 1],
    )

    fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
    image = ax.imshow(cm)
    ax.set_title(
        "Test Confusion Matrix",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Predicted Class", fontsize=11)
    ax.set_ylabel("True Class", fontsize=11)
    ax.set_xticks([0, 1], labels=["Real", "Fake"])
    ax.set_yticks([0, 1], labels=["Real", "Fake"])

    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
                fontsize=14,
            )

    fig.colorbar(image, ax=ax)
    save_figure(
        fig,
        figures_dir / "test_confusion_matrix.png",
        min_short_edge,
    )

    if len(np.unique(labels)) == 2:
        fpr, tpr, _ = roc_curve(labels, probabilities)
        auc = roc_auc_score(labels, probabilities)

        fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
        ax.plot(
            fpr,
            tpr,
            linewidth=2,
            label=f"Swin V2 Tiny (AUC = {auc:.4f})",
        )
        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            linewidth=1.5,
            label="Random Classifier",
        )
        ax.set_title(
            "Test ROC Curve",
            fontsize=14,
            fontweight="bold",
        )
        ax.set_xlabel(
            "False Positive Rate",
            fontsize=11,
        )
        ax.set_ylabel(
            "True Positive Rate",
            fontsize=11,
        )
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.01)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.25)
        save_figure(
            fig,
            figures_dir / "test_roc_curve.png",
            min_short_edge,
        )

    precision, recall, _ = precision_recall_curve(
        labels,
        probabilities,
    )
    ap = average_precision_score(
        labels,
        probabilities,
    )

    fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
    ax.plot(
        recall,
        precision,
        linewidth=2,
        label=f"Swin V2 Tiny (AP = {ap:.4f})",
    )
    ax.set_title(
        "Test Precision-Recall Curve",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Recall", fontsize=11)
    ax.set_ylabel("Precision", fontsize=11)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.01)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)
    save_figure(
        fig,
        figures_dir / "test_precision_recall_curve.png",
        min_short_edge,
    )

Writing /content/mouth_swinv2_experiment/src/deepfake_roi/evaluation/plots.py


In [11]:
# ============================================================
# 3. IMPORT MODULAR PACKAGE
# ============================================================
import sys

sys.path.insert(0, str(SRC_ROOT))

from deepfake_roi.data.dataset import MouthROIDataset, build_transforms
from deepfake_roi.evaluation.plots import generate_all_figures
from deepfake_roi.models.swin import SwinV2TinyBinaryClassifier
from deepfake_roi.training.engine import (
    assert_finite_gradients,
    assert_finite_tensor,
    compute_metrics,
    run_epoch,
    select_best_f1_threshold,
)
from deepfake_roi.utils.io import (
    CHECKPOINT_REQUIRED_KEYS,
    atomic_save_checkpoint,
    hash_source_tree,
    load_checkpoint,
    save_json_atomic,
)
from deepfake_roi.utils.repro import (
    capture_rng_state,
    restore_rng_state,
    seed_everything,
    seed_worker,
)

print("Modular package import: PASSED")

Modular package import: PASSED


In [12]:
# ============================================================
# 4. SINGLE SOURCE OF TRUTH CONFIG
# ============================================================
from typing import Any, Dict

CONFIG: Dict[str, Any] = {
    "experiment": {
        "region": "mouth",
        "model_name": "swinv2_tiny",
        "seed": 42,
        "notebook_version": "v3_stabilized",

        # CPU'da tamamlanan 1. epoch'tan devam ediyoruz.
        "mode": "resume",
        "resume_run_id": "20260807_2235_mouth_swinv2_tiny_seed42",
    },

    "paths": {
        "project_root_candidates": [
            "/content/drive/MyDrive/AISC Çalışmalar",
            "/content/drive/MyDrive/AISC Çalışmalar",
            "/content/drive/MyDrive/AISC DeepFake Çalışmaları",
            "/content/drive/.shortcut-targets-by-id/1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1/AISC DeepFake Çalışmaları",
        ],

        "mouth_roi_relative_candidates": [
            "Deneyler/Dilara/Deney 1/mouth_roi_output",
            "Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output",
            "Deneyler/Dilara/Deney 1/Agiz/mouth_roi_output",
        ],

        "results_relative": "Deneyler/Dilara/Deney 1/Sonuçlar",
    },

    "data": {
        "metadata_filename": "metadata.csv",

        "image_column": "mouth_path",

        "image_column_candidates": [
            "mouth_path",
            "mouth_roi_path",
            "combined_mouth_path",
            "output_path",
        ],

        "sample_id_column": "sample_id",
        "video_id_column": "video_id",
        "label_column": "label",
        "split_column": "split",
        "status_column": "status",

        "accepted_status": ["success"],

        "allowed_labels": ["real", "fake"],
        "allowed_splits": ["train", "val", "test"],

        "image_size": 224,
        "batch_size": 32,
        "num_workers": 2,
        "pin_memory": True,

        "strict_ssot_schema": False,
    },

    "normalization": {
        "source": "pretrained_weights",
        "weights": "Swin_V2_T_Weights.IMAGENET1K_V1",
        "rationale": (
            "ImageNet-pretrained Swin V2'nin beklediği normalizasyon kullanılır. "
            "Project train/val/test verisinden mean/std öğrenilmez; dolayısıyla "
            "validation/test istatistiği sızıntısı oluşmaz."
        ),
    },

    "model": {
        "pretrained": True,
        "dropout": 0.20,
    },

    "training": {
        "epochs": 20,
        "backbone_lr": 1e-5,
        "head_lr": 1e-4,
        "weight_decay": 1e-2,
        "gradient_clip_norm": 1.0,

        "mixed_precision": False,
        "amp_overflow_abort_after": 8,

        "early_stopping_patience": 6,
        "monitor_metric": "roc_auc",
        "fallback_metric": "f1",
        "keep_last_n_epoch_checkpoints": 3,
    },

    "augmentation": {
        "horizontal_flip_probability": 0.50,
        "rotation_degrees": 5,
        "brightness": 0.10,
        "contrast": 0.10,
        "saturation": 0.05,
        "hue": 0.02,
    },

    "quality": {
        "smoke_test_batches": 2,
        "min_figure_short_edge_px": 600,
        "checkpoint_continuity_atol": 1e-7,
        "checkpoint_continuity_rtol": 1e-6,
    },
}

In [13]:
# ============================================================
# 5. RESOLVE PATHS + NEW/RESUME RUN
# ============================================================
import json
import os
import random
import shutil
import subprocess
from datetime import datetime
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import PIL
import sklearn
import torch
import torchvision
import yaml

def first_existing_path(candidates):
    existing = [Path(p) for p in candidates if Path(p).exists()]
    if not existing:
        raise FileNotFoundError(
            "AISC DeepFake Çalışmaları klasörü bulunamadı.\n"
            "Kontrol edilen yollar:\n- " + "\n- ".join(candidates)
        )
    return existing[0]

PROJECT_ROOT = first_existing_path(CONFIG["paths"]["project_root_candidates"])
mouth_roi_candidates = [
    PROJECT_ROOT / rel
    for rel in CONFIG["paths"]["mouth_roi_relative_candidates"]
]
existing_mouth_roots = [p for p in mouth_roi_candidates if p.is_dir()]
if not existing_mouth_roots:
    raise FileNotFoundError(
        "Ağız ROI klasörü bulunamadı. Kontrol edilen yollar:\n- "
        + "\n- ".join(str(p) for p in mouth_roi_candidates)
    )
MOUTH_ROI_ROOT = existing_mouth_roots[0]
METADATA_PATH = MOUTH_ROI_ROOT / CONFIG["data"]["metadata_filename"]
RESULTS_ROOT = PROJECT_ROOT / CONFIG["paths"]["results_relative"]

if not METADATA_PATH.is_file():
    raise FileNotFoundError(METADATA_PATH)

metadata_columns = set(pd.read_csv(METADATA_PATH, nrows=0).columns)
image_candidates = CONFIG["data"]["image_column_candidates"]
matched_image_columns = [c for c in image_candidates if c in metadata_columns]
if not matched_image_columns:
    raise ValueError(
        "Metadata içinde ağız ROI yolu sütunu bulunamadı. "
        f"Beklenenlerden biri: {image_candidates}; mevcut: {sorted(metadata_columns)}"
    )
CONFIG["data"]["image_column"] = matched_image_columns[0]

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

mode = str(CONFIG["experiment"]["mode"]).lower().strip()
if mode not in {"new", "resume"}:
    raise ValueError("experiment.mode must be 'new' or 'resume'.")

SEED = int(CONFIG["experiment"]["seed"])

if mode == "new":
    run_id = (
        f"{datetime.now():%Y%m%d_%H%M}_"
        f"{CONFIG['experiment']['region']}_"
        f"{CONFIG['experiment']['model_name']}_seed{SEED}"
    )
    RUN_DIR = RESULTS_ROOT / run_id

    if RUN_DIR.exists():
        suffix = 2
        while (RESULTS_ROOT / f"{run_id}_r{suffix}").exists():
            suffix += 1
        run_id = f"{run_id}_r{suffix}"
        RUN_DIR = RESULTS_ROOT / run_id

    RUN_DIR.mkdir(parents=True, exist_ok=False)

else:
    resume_run_id = CONFIG["experiment"]["resume_run_id"]
    if not resume_run_id:
        raise ValueError(
            "Resume mode seçildi fakat experiment.resume_run_id boş."
        )

    run_id = str(resume_run_id)
    RUN_DIR = RESULTS_ROOT / run_id

    if not RUN_DIR.is_dir():
        raise FileNotFoundError(
            f"Resume run directory bulunamadı: {RUN_DIR}"
        )

CHECKPOINT_DIR = RUN_DIR / "checkpoints"
METRICS_DIR = RUN_DIR / "metrics"
PREDICTIONS_DIR = RUN_DIR / "predictions"
FIGURES_DIR = RUN_DIR / "figures"
AUDIT_DIR = RUN_DIR / "audit"
LOG_DIR = RUN_DIR / "logs"
SOURCE_DIR = RUN_DIR / "source_snapshot"

for d in [
    CHECKPOINT_DIR,
    METRICS_DIR,
    PREDICTIONS_DIR,
    FIGURES_DIR,
    AUDIT_DIR,
    LOG_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Resolved config: resume sırasında eski run'ın configini ezmeyelim.
resolved_name = (
    "config_resolved.yaml"
    if mode == "new"
    else f"config_resume_{datetime.now():%Y%m%d_%H%M%S}.yaml"
)
with (RUN_DIR / resolved_name).open("w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, sort_keys=False, allow_unicode=True)

print("MODE         :", mode)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("METADATA     :", METADATA_PATH)
print("RUN_DIR      :", RUN_DIR)

MODE         : resume
PROJECT_ROOT : /content/drive/MyDrive/AISC DeepFake Çalışmaları
METADATA     : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output/metadata.csv
RUN_DIR      : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/20260807_2235_mouth_swinv2_tiny_seed42


In [14]:
# ============================================================
# 6. ENVIRONMENT LOCK + SOURCE SNAPSHOT/HASH
# ============================================================
# Exact runtime versions are locked for reproducibility.
# This includes torch/torchvision without modifying Colab's CUDA stack.

import importlib.metadata as md

packages_to_lock = [
    "torch",
    "torchvision",
    "numpy",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "Pillow",
    "PyYAML",
    "tqdm",
]

lock_lines = []
for package in packages_to_lock:
    try:
        version = md.version(package)
        lock_lines.append(f"{package}=={version}")
    except md.PackageNotFoundError:
        raise RuntimeError(f"Required package missing: {package}")

requirements_lock = RUN_DIR / "requirements_runtime.lock.txt"
requirements_lock.write_text(
    "\n".join(lock_lines) + "\n",
    encoding="utf-8",
)

environment_manifest = {
    "run_id": run_id,
    "python": sys.version,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "seed": SEED,
    "locked_requirements_file": str(requirements_lock),
}
save_json_atomic(
    environment_manifest,
    RUN_DIR / "environment.json",
)

# Source snapshot only on NEW run; resume keeps original snapshot intact.
if mode == "new":
    shutil.copytree(
        SRC_ROOT / "deepfake_roi",
        SOURCE_DIR / "deepfake_roi",
        dirs_exist_ok=False,
    )

source_files = list(
    (SOURCE_DIR / "deepfake_roi").rglob("*.py")
) if SOURCE_DIR.exists() else []

source_hashes = hash_source_tree(source_files)

save_json_atomic(
    {
        "run_id": run_id,
        "files": source_hashes,
    },
    RUN_DIR / "source_hash_manifest.json",
)

print(requirements_lock.read_text())
print("Source snapshot files:", len(source_files))

torch==2.11.0+cu128
torchvision==0.26.0+cu128
numpy==2.0.2
pandas==2.2.2
scikit-learn==1.6.1
matplotlib==3.10.0
Pillow==11.3.0
PyYAML==6.0.3
tqdm==4.67.3

Source snapshot files: 12


In [15]:
# ============================================================
# 7. REPRODUCIBILITY
# ============================================================
seed_everything(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE: cuda
GPU: Tesla T4


In [17]:
# ============================================================
# 8. METADATA SCHEMA + ACCOUNTING + LEAKAGE AUDIT
# ============================================================
metadata = pd.read_csv(METADATA_PATH)

D = CONFIG["data"]

sample_col = D["sample_id_column"]
video_col = D["video_id_column"]
label_col = D["label_column"]
split_col = D["split_column"]
status_col = D["status_column"]
image_col = D["image_column"]

training_required_columns = {
    sample_col,
    video_col,
    label_col,
    split_col,
    status_col,
    image_col,
}

missing_training = training_required_columns.difference(metadata.columns)

if missing_training:
    raise ValueError(
        "Training-required metadata columns missing: "
        f"{sorted(missing_training)}"
    )

# Takım standardında beklenen tam metadata şeması.
# Metadata'da bulunmayan alanlar uydurulmaz; yalnızca raporlanır.
ssot_expected = {
    "sample_id",
    "source_video",
    "frame_index",
    "face_index",
    "roi_state",
    "label",
    "split",
    "status",
    "skip_reason",
    "sha256",
    "output_path",
    "run_id",
}

# Mevcut metadata'daki eş anlamlı sütunlar.
ssot_aliases = {
    "source_video": "video_id",
    "face_index": "face_id",
    "output_path": image_col,
    "skip_reason": "error",
}

effective_present = set(metadata.columns)

for canonical, alias in ssot_aliases.items():
    if alias in metadata.columns:
        effective_present.add(canonical)

ssot_missing = sorted(ssot_expected - effective_present)

schema_audit = {
    "training_required_columns": sorted(training_required_columns),
    "training_required_missing": sorted(missing_training),
    "ssot_expected_columns": sorted(ssot_expected),
    "ssot_aliases_used": ssot_aliases,
    "ssot_missing_after_aliases": ssot_missing,
    "strict_ssot_schema": bool(D["strict_ssot_schema"]),
    "note": (
        "This notebook consumes upstream metadata. "
        "Missing SSOT fields are reported, not invented."
    ),
}

save_json_atomic(
    schema_audit,
    AUDIT_DIR / "metadata_schema_audit.json",
)

if D["strict_ssot_schema"] and ssot_missing:
    raise ValueError(
        "Strict SSOT schema enabled; missing fields: "
        f"{ssot_missing}"
    )

metadata = metadata.copy()

# Metin alanlarını standartlaştır.
for col in [
    sample_col,
    video_col,
    label_col,
    split_col,
    status_col,
]:
    metadata[col] = (
        metadata[col]
        .astype(str)
        .str.strip()
    )

metadata[label_col] = metadata[label_col].str.lower()
metadata[split_col] = metadata[split_col].str.lower()
metadata[status_col] = metadata[status_col].str.lower()

raw_count = len(metadata)

accepted_status = {
    str(x).lower()
    for x in D["accepted_status"]
}

status_counts = (
    metadata[status_col]
    .value_counts(dropna=False)
    .to_dict()
)

# Yalnızca başarılı ağız ROI kayıtlarını kullan.
metadata = metadata[
    metadata[status_col].isin(accepted_status)
].copy()

if metadata.empty:
    raise RuntimeError(
        "No accepted records. "
        f"status distribution={status_counts}"
    )

allowed_labels = set(D["allowed_labels"])
allowed_splits = set(D["allowed_splits"])

unexpected_labels = (
    set(metadata[label_col].unique())
    - allowed_labels
)

unexpected_splits = (
    set(metadata[split_col].unique())
    - allowed_splits
)

if unexpected_labels:
    raise ValueError(
        f"Unexpected labels: {unexpected_labels}"
    )

if unexpected_splits:
    raise ValueError(
        f"Unexpected splits: {unexpected_splits}"
    )

missing_splits = (
    allowed_splits
    - set(metadata[split_col].unique())
)

if missing_splits:
    raise ValueError(
        f"Required splits missing: {missing_splits}"
    )

# Her split hem real hem fake içermeli.
for split_name in sorted(allowed_splits):
    present_labels = set(
        metadata.loc[
            metadata[split_col] == split_name,
            label_col,
        ].unique()
    )

    missing_labels = allowed_labels - present_labels

    if missing_labels:
        raise ValueError(
            f"Split {split_name!r} is missing labels: "
            f"{sorted(missing_labels)}"
        )

# sample_id tekrar kontrolü.
dup_mask = metadata[sample_col].duplicated(keep=False)

if dup_mask.any():
    duplicate_file = (
        AUDIT_DIR / "duplicate_sample_ids.csv"
    )

    metadata.loc[dup_mask].to_csv(
        duplicate_file,
        index=False,
    )

    raise ValueError(
        f"Duplicate sample_id detected: {duplicate_file}"
    )


def resolve_roi_path(value):
    """
    Metadata içindeki ağız ROI yolunu çözer.

    Metadata eski Google Drive/kısayol kökünü içeriyorsa,
    mouth_roi_output sonrasındaki göreli bölümü güncel
    MOUTH_ROI_ROOT klasörüne bağlar.
    """
    if pd.isna(value) or not str(value).strip():
        raise ValueError(
            "Empty mouth ROI path detected."
        )

    original_path = Path(str(value).strip())

    # Metadata içindeki yol mevcut ortamda çalışıyorsa aynen kullan.
    if original_path.is_file():
        return original_path

    # Göreli yol verilmişse güncel ağız ROI köküne bağla.
    if not original_path.is_absolute():
        return MOUTH_ROI_ROOT / original_path

    # Eski Drive veya kısayol kökü kullanılmışsa yeniden eşle.
    parts = original_path.parts

    if "mouth_roi_output" in parts:
        marker_index = parts.index("mouth_roi_output")

        relative_part = Path(
            *parts[marker_index + 1:]
        )

        remapped_path = (
            MOUTH_ROI_ROOT / relative_part
        )

        return remapped_path

    # Yeniden eşleştirilemeyen yol daha sonra eksik dosya
    # denetiminde açıkça raporlanır.
    return original_path


# Her kayıt için kullanılacak gerçek görüntü yolunu belirle.
metadata["_resolved_image_path"] = (
    metadata[image_col]
    .map(
        lambda value: str(
            resolve_roi_path(value)
        )
    )
)

# Bütün ağız ROI görüntülerinin gerçekten var olduğunu doğrula.
exists_mask = metadata["_resolved_image_path"].map(
    lambda path: Path(path).is_file()
)

if not exists_mask.all():
    missing_file = (
        AUDIT_DIR / "missing_images.csv"
    )

    metadata.loc[~exists_mask].to_csv(
        missing_file,
        index=False,
    )

    raise FileNotFoundError(
        f"{int((~exists_mask).sum())} ROI files missing. "
        f"Audit: {missing_file}"
    )

# Aynı video_id yalnızca tek etikete sahip olmalı.
video_label_counts = (
    metadata.groupby(video_col)[label_col].nunique()
)

if (video_label_counts > 1).any():
    bad_ids = video_label_counts[
        video_label_counts > 1
    ].index

    bad_file = (
        AUDIT_DIR / "video_label_conflicts.csv"
    )

    metadata[
        metadata[video_col].isin(bad_ids)
    ].to_csv(
        bad_file,
        index=False,
    )

    raise ValueError(
        f"Video label conflict: {bad_file}"
    )

# Aynı video_id yalnızca tek splitte bulunmalı.
video_split_counts = (
    metadata.groupby(video_col)[split_col].nunique()
)

if (video_split_counts > 1).any():
    bad_ids = video_split_counts[
        video_split_counts > 1
    ].index

    bad_file = (
        AUDIT_DIR / "video_split_conflicts.csv"
    )

    metadata[
        metadata[video_col].isin(bad_ids)
    ].to_csv(
        bad_file,
        index=False,
    )

    raise ValueError(
        f"Video split conflict: {bad_file}"
    )

# Train, validation ve test video kümeleri.
train_videos = set(
    metadata.loc[
        metadata[split_col] == "train",
        video_col,
    ]
)

val_videos = set(
    metadata.loc[
        metadata[split_col] == "val",
        video_col,
    ]
)

test_videos = set(
    metadata.loc[
        metadata[split_col] == "test",
        video_col,
    ]
)

# Splitler arasında video_id çakışması olmamalı.
assert train_videos.isdisjoint(val_videos)
assert train_videos.isdisjoint(test_videos)
assert val_videos.isdisjoint(test_videos)

# video_id değerlerinin gerçek kaynak video kimliği yerine
# fake_train / real_test gibi kaba kovalar olup olmadığını kontrol et.
expected_bucket_id = (
    metadata[label_col].astype(str)
    + "_"
    + metadata[split_col].astype(str)
)

coarse_video_fraction = float(
    (
        metadata[video_col]
        .astype(str)
        .str.lower()
        ==
        expected_bucket_id.str.lower()
    ).mean()
)

video_identity_reliable = (
    coarse_video_fraction < 0.80
)

save_json_atomic(
    {
        "video_identity_reliable_for_video_level_metrics":
            video_identity_reliable,
        "coarse_label_split_bucket_fraction":
            coarse_video_fraction,
        "interpretation": (
            "If false, video_id appears to encode only "
            "label/split buckets. Frame-level training may "
            "continue, but true source-video-level leakage "
            "cannot be independently verified and video-level "
            "metrics must not be interpreted as "
            "per-original-video performance."
        ),
    },
    AUDIT_DIR / "video_identity_audit.json",
)

# Veri dağılım tablosu.
summary_rows = []

for split_name in ["train", "val", "test"]:
    split_df = metadata[
        metadata[split_col] == split_name
    ]

    for label_name in ["real", "fake"]:
        label_df = split_df[
            split_df[label_col] == label_name
        ]

        summary_rows.append(
            {
                "split": split_name,
                "label": label_name,
                "frames": len(label_df),
                "videos": (
                    label_df[video_col].nunique()
                ),
            }
        )

dataset_summary = pd.DataFrame(summary_rows)

dataset_summary.to_csv(
    AUDIT_DIR / "dataset_summary.csv",
    index=False,
)

# Doğrulanmış eğitim manifestini kaydet.
metadata.to_csv(
    AUDIT_DIR / "validated_training_manifest.csv",
    index=False,
)

# Leakage ve veri muhasebesi raporu.
save_json_atomic(
    {
        "raw_rows": raw_count,
        "accepted_rows": len(metadata),
        "status_counts_before_filter": status_counts,
        "train_video_count": len(train_videos),
        "val_video_count": len(val_videos),
        "test_video_count": len(test_videos),
        "train_val_overlap": len(
            train_videos & val_videos
        ),
        "train_test_overlap": len(
            train_videos & test_videos
        ),
        "val_test_overlap": len(
            val_videos & test_videos
        ),
        "mouth_roi_root": str(MOUTH_ROI_ROOT),
        "metadata_path": str(METADATA_PATH),
        "image_column": image_col,
        "path_remapping_enabled": True,
    },
    AUDIT_DIR / "split_leakage_report.json",
)

display(dataset_summary)

print(
    "Metadata/schema/leakage audit: PASSED"
)

print(
    "SSOT missing (reported, not fabricated):",
    ssot_missing,
)

print(
    "Accepted mouth ROI records:",
    len(metadata),
)

print(
    "Mouth ROI root:",
    MOUTH_ROI_ROOT,
)

,split,label,frames,videos
0,train,real,1197,1172
1,train,fake,1192,1138
2,val,real,155,150
3,val,fake,141,137
4,test,real,146,145
5,test,fake,156,147


Metadata/schema/leakage audit: PASSED
SSOT missing (reported, not fabricated): ['frame_index', 'roi_state', 'run_id', 'sha256']
Accepted mouth ROI records: 2987
Mouth ROI root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output


In [18]:
# ============================================================
# 9. NORMALIZATION AUDIT + DATASET/DATALOADERS
# ============================================================
train_transform, eval_transform, normalization_info = build_transforms(
    image_size=int(D["image_size"]),
    augmentation=CONFIG["augmentation"],
)

normalization_audit = {
    **normalization_info,
    "config_source": CONFIG["normalization"]["source"],
    "rationale": CONFIG["normalization"]["rationale"],
}

save_json_atomic(
    normalization_audit,
    AUDIT_DIR / "normalization_audit.json",
)

if normalization_info["uses_validation_or_test_statistics"]:
    raise RuntimeError(
        "Normalization leakage detected."
    )

train_df = metadata[
    metadata[split_col] == "train"
].copy()
val_df = metadata[
    metadata[split_col] == "val"
].copy()
test_df = metadata[
    metadata[split_col] == "test"
].copy()

for name, frame in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
]:
    if frame.empty:
        raise RuntimeError(f"{name} split is empty.")

train_dataset = MouthROIDataset(
    train_df,
    train_transform,
    sample_col,
    video_col,
    label_col,
)
val_dataset = MouthROIDataset(
    val_df,
    eval_transform,
    sample_col,
    video_col,
    label_col,
)
test_dataset = MouthROIDataset(
    test_df,
    eval_transform,
    sample_col,
    video_col,
    label_col,
)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

loader_kwargs = {
    "batch_size": int(D["batch_size"]),
    "num_workers": int(D["num_workers"]),
    "pin_memory": (
        bool(D["pin_memory"])
        and DEVICE.type == "cuda"
    ),
    "worker_init_fn": seed_worker,
    "generator": loader_generator,
    "persistent_workers": (
        int(D["num_workers"]) > 0
    ),
}

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    shuffle=True,
    drop_last=False,
    **loader_kwargs,
)

# Validation/test için farklı generator gerekmez; shuffle=False.
eval_loader_kwargs = dict(loader_kwargs)
eval_loader_kwargs["generator"] = None

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    shuffle=False,
    drop_last=False,
    **eval_loader_kwargs,
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    shuffle=False,
    drop_last=False,
    **eval_loader_kwargs,
)

print("Train / Val / Test frames:",
      len(train_df), len(val_df), len(test_df))
print("Normalization:", normalization_audit)

Train / Val / Test frames: 2389 296 302
Normalization: {'source': 'Swin_V2_T_Weights.IMAGENET1K_V1', 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225], 'learned_from_project_data': False, 'uses_validation_or_test_statistics': False, 'config_source': 'pretrained_weights', 'rationale': "ImageNet-pretrained Swin V2'nin beklediği normalizasyon kullanılır. Project train/val/test verisinden mean/std öğrenilmez; dolayısıyla validation/test istatistiği sızıntısı oluşmaz."}


In [19]:
# ============================================================
# 10. MODEL + TRAIN-ONLY LOSS WEIGHT + OPTIMIZER
# ============================================================
model = SwinV2TinyBinaryClassifier(
    pretrained=bool(CONFIG["model"]["pretrained"]),
    dropout=float(CONFIG["model"]["dropout"]),
).to(DEVICE)

train_counts = train_df[label_col].value_counts()

num_real = int(train_counts.get("real", 0))
num_fake = int(train_counts.get("fake", 0))

if num_real == 0 or num_fake == 0:
    raise RuntimeError(
        "Training split must contain both classes."
    )

# Train-only statistic.
pos_weight_value = num_real / num_fake
pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=DEVICE,
)

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

T = CONFIG["training"]

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.backbone.parameters(),
            "lr": float(T["backbone_lr"]),
        },
        {
            "params": model.classifier.parameters(),
            "lr": float(T["head_lr"]),
        },
    ],
    weight_decay=float(T["weight_decay"]),
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=int(T["epochs"]),
    eta_min=float(T["backbone_lr"]) * 0.1,
)

AMP_ENABLED = (
    bool(T["mixed_precision"])
    and DEVICE.type == "cuda"
)

scaler_device = "cuda" if DEVICE.type == "cuda" else "cpu"
scaler = torch.amp.GradScaler(
    scaler_device,
    enabled=AMP_ENABLED,
)

print("Train real:", num_real)
print("Train fake:", num_fake)
print("pos_weight:", pos_weight_value)
print("AMP:", AMP_ENABLED)

Downloading: "https://download.pytorch.org/models/swin_v2_t-b137f0e2.pth" to /root/.cache/torch/hub/checkpoints/swin_v2_t-b137f0e2.pth


100%|██████████| 109M/109M [00:00<00:00, 170MB/s] 


Train real: 1197
Train fake: 1192
pos_weight: 1.0041946308724832
AMP: False


In [20]:
# ============================================================
# 11. INPUT SANITY + FP32 SMOKE TEST
# ============================================================
import copy
import torch

# Numerical input sanity gate before any optimizer update.
diagnostic_batch = next(iter(train_loader))
diagnostic_images = diagnostic_batch["image"]
diagnostic_labels = diagnostic_batch["label"]

input_report = {
    "shape": list(diagnostic_images.shape),
    "dtype": str(diagnostic_images.dtype),
    "min": float(diagnostic_images.min().item()),
    "max": float(diagnostic_images.max().item()),
    "mean": float(diagnostic_images.mean().item()),
    "std": float(diagnostic_images.std().item()),
    "images_finite": bool(torch.isfinite(diagnostic_images).all().item()),
    "labels_finite": bool(torch.isfinite(diagnostic_labels).all().item()),
}

save_json_atomic(
    input_report,
    AUDIT_DIR / "input_numerical_sanity.json",
)

if not input_report["images_finite"]:
    raise FloatingPointError("NaN/Inf detected in input images.")
if not input_report["labels_finite"]:
    raise FloatingPointError("NaN/Inf detected in labels.")

print("Input numerical sanity: PASSED")
print(input_report)


def run_smoke_test(
    base_model,
    loader,
    max_batches: int,
):
    # The quality gate intentionally runs in FP32 regardless of the later
    # optional AMP setting. This distinguishes model/data instability from
    # mixed-precision overflow.
    smoke_model = copy.deepcopy(base_model).to(DEVICE)
    smoke_optimizer = torch.optim.AdamW(
        smoke_model.parameters(),
        lr=1e-6,
    )
    smoke_model.train()

    completed = 0

    for batch in loader:
        images = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        assert_finite_tensor(images, "smoke input")
        assert_finite_tensor(labels, "smoke labels")

        smoke_optimizer.zero_grad(set_to_none=True)

        # No autocast here: strict FP32 diagnostic.
        logits = smoke_model(images)
        assert_finite_tensor(logits, "smoke logits")

        loss = criterion(logits, labels)
        assert_finite_tensor(loss, "smoke test loss")

        loss.backward()
        assert_finite_gradients(smoke_model)

        torch.nn.utils.clip_grad_norm_(
            smoke_model.parameters(),
            max_norm=float(T["gradient_clip_norm"]),
            error_if_nonfinite=True,
        )
        smoke_optimizer.step()

        completed += 1
        print(
            f"FP32 smoke batch {completed}: "
            f"loss={loss.item():.8f}"
        )

        if completed >= max_batches:
            break

    del smoke_model
    del smoke_optimizer

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    if completed < max_batches:
        raise RuntimeError(
            "FP32 smoke test could not complete."
        )

    print("FP32 smoke test: PASSED")


run_smoke_test(
    model,
    train_loader,
    int(CONFIG["quality"]["smoke_test_batches"]),
)

Input numerical sanity: PASSED
{'shape': [32, 3, 224, 224], 'dtype': 'torch.float32', 'min': -2.1179039478302, 'max': 2.640000104904175, 'mean': -0.9128353595733643, 'std': 1.150808572769165, 'images_finite': True, 'labels_finite': True}
FP32 smoke batch 1: loss=0.72301793
FP32 smoke batch 2: loss=0.77138001
FP32 smoke test: PASSED


In [21]:
# ============================================================
# 12. CHECKPOINT BUILD + CONTINUITY QUALITY GATE
# ============================================================

import torch

BEST_CHECKPOINT = CHECKPOINT_DIR / "best.ckpt"
LAST_CHECKPOINT = CHECKPOINT_DIR / "last.ckpt"

def build_checkpoint_state(
    *,
    epoch,
    best_metric_score,
    best_threshold,
    history,
    best_epoch,
    epochs_without_improvement,
):
    return {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_metric_score": float(best_metric_score),
        "best_threshold": float(best_threshold),
        "best_epoch": int(best_epoch),
        "epochs_without_improvement": int(epochs_without_improvement),
        "history": history,
        "config": CONFIG,
        "run_id": run_id,
        "rng_state": capture_rng_state(),
        "loader_generator_state": loader_generator.get_state(),
    }


def checkpoint_continuity_test():
    # Deterministic validation batch: no augmentation, model.eval() disables dropout.
    batch = next(iter(val_loader))
    images = batch["image"].to(DEVICE)
    labels = batch["label"].to(DEVICE)

    original_training_mode = model.training
    model.eval()

    with torch.no_grad():
        logits_before = model(images)
        loss_before = criterion(
            logits_before,
            labels,
        )

    temporary = (
        CHECKPOINT_DIR /
        "_continuity_test.ckpt"
    )

    state = build_checkpoint_state(
        epoch=0,
        best_metric_score=-1.0,
        best_threshold=0.5,
        history=[],
        best_epoch=0,
        epochs_without_improvement=0,
    )

    atomic_save_checkpoint(
        state,
        temporary,
    )

    fresh_model = SwinV2TinyBinaryClassifier(
        pretrained=False,
        dropout=float(CONFIG["model"]["dropout"]),
    ).to(DEVICE)

    fresh_optimizer = torch.optim.AdamW(
        [
            {
                "params": fresh_model.backbone.parameters(),
                "lr": float(T["backbone_lr"]),
            },
            {
                "params": fresh_model.classifier.parameters(),
                "lr": float(T["head_lr"]),
            },
        ],
        weight_decay=float(T["weight_decay"]),
    )

    fresh_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        fresh_optimizer,
        T_max=int(T["epochs"]),
        eta_min=float(T["backbone_lr"]) * 0.1,
    )

    fresh_scaler = torch.amp.GradScaler(
        "cuda" if DEVICE.type == "cuda" else "cpu",
        enabled=AMP_ENABLED,
    )

    loaded = load_checkpoint(
        temporary,
        DEVICE,
    )

    fresh_model.load_state_dict(
        loaded["model_state_dict"]
    )
    fresh_optimizer.load_state_dict(
        loaded["optimizer_state_dict"]
    )
    fresh_scheduler.load_state_dict(
        loaded["scheduler_state_dict"]
    )
    fresh_scaler.load_state_dict(
        loaded["scaler_state_dict"]
    )

    fresh_model.eval()

    with torch.no_grad():
        logits_after = fresh_model(images)
        loss_after = criterion(
            logits_after,
            labels,
        )

    atol = float(
        CONFIG["quality"]["checkpoint_continuity_atol"]
    )
    rtol = float(
        CONFIG["quality"]["checkpoint_continuity_rtol"]
    )

    if not torch.allclose(
        logits_before,
        logits_after,
        atol=atol,
        rtol=rtol,
    ):
        raise RuntimeError(
            "Checkpoint continuity FAILED: logits mismatch."
        )

    if not torch.allclose(
        loss_before,
        loss_after,
        atol=atol,
        rtol=rtol,
    ):
        raise RuntimeError(
            "Checkpoint continuity FAILED: loss mismatch."
        )

    report = {
        "passed": True,
        "loss_before": float(loss_before.item()),
        "loss_after": float(loss_after.item()),
        "absolute_loss_difference": float(
            abs(loss_before.item() - loss_after.item())
        ),
        "atol": atol,
        "rtol": rtol,
    }

    save_json_atomic(
        report,
        AUDIT_DIR / "checkpoint_continuity_test.json",
    )

    temporary.unlink()

    if original_training_mode:
        model.train()

    del fresh_model
    del fresh_optimizer
    del fresh_scheduler
    del fresh_scaler

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    print("Checkpoint continuity test: PASSED")
    print(report)


checkpoint_continuity_test()


Checkpoint continuity test: PASSED
{'passed': True, 'loss_before': 0.798557698726654, 'loss_after': 0.798557698726654, 'absolute_loss_difference': 0.0, 'atol': 1e-07, 'rtol': 1e-06}


In [23]:
# ============================================================
# 13. TRUE RESUME LOGIC — CPU -> GPU SAFE
# ============================================================

import numpy as np
import torch

history = []
best_metric_score = float("-inf")
best_threshold = 0.5
best_epoch = 0
epochs_without_improvement = 0
start_epoch = 1


def as_cpu_byte_tensor(value):
    """
    CPU'da kaydedilmiş RNG durumunu yeni PyTorch/T4
    oturumunun beklediği torch.ByteTensor biçimine dönüştürür.
    """
    if isinstance(value, torch.Tensor):
        return value.detach().to(
            device="cpu",
            dtype=torch.uint8,
        ).contiguous()

    return torch.as_tensor(
        value,
        dtype=torch.uint8,
        device="cpu",
    ).contiguous()


if mode == "resume":

    if not LAST_CHECKPOINT.is_file():
        raise FileNotFoundError(
            f"Resume requested but last.ckpt missing: "
            f"{LAST_CHECKPOINT}"
        )

    print("Checkpoint yükleniyor:", LAST_CHECKPOINT)

    resume_state = load_checkpoint(
        LAST_CHECKPOINT,
        DEVICE,
    )

    # --------------------------------------------------------
    # Deney kimliği kontrolü
    # --------------------------------------------------------
    old_config = resume_state["config"]

    identity_fields = [
        ("experiment", "region"),
        ("experiment", "model_name"),
        ("experiment", "seed"),
        ("data", "image_size"),
        ("data", "image_column"),
    ]

    mismatches = []

    for section, key in identity_fields:
        old_value = old_config[section][key]
        new_value = CONFIG[section][key]

        if old_value != new_value:
            mismatches.append(
                f"{section}.{key}: "
                f"checkpoint={old_value!r}, "
                f"current={new_value!r}"
            )

    if mismatches:
        raise ValueError(
            "Resume config identity mismatch:\n"
            + "\n".join(mismatches)
        )

    print("Checkpoint experiment identity: PASSED")

    # --------------------------------------------------------
    # Model ve eğitim durumlarını geri yükle
    # --------------------------------------------------------
    model.load_state_dict(
        resume_state["model_state_dict"]
    )

    optimizer.load_state_dict(
        resume_state["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        resume_state["scheduler_state_dict"]
    )

    scaler.load_state_dict(
        resume_state["scaler_state_dict"]
    )

    # Optimizer tensörlerinin seçilen cihazda olduğundan emin ol
    for optimizer_state in optimizer.state.values():
        for key, value in optimizer_state.items():
            if isinstance(value, torch.Tensor):
                optimizer_state[key] = value.to(DEVICE)

    # --------------------------------------------------------
    # CPU -> GPU geçişine dayanıklı RNG geri yükleme
    # --------------------------------------------------------
    saved_rng_state = resume_state.get("rng_state")

    if saved_rng_state is not None:
        try:
            saved_rng_state = dict(saved_rng_state)

            if saved_rng_state.get("torch_rng_state") is not None:
                saved_rng_state["torch_rng_state"] = (
                    as_cpu_byte_tensor(
                        saved_rng_state["torch_rng_state"]
                    )
                )

            cuda_states = saved_rng_state.get("cuda_rng_state")

            if cuda_states is not None:
                saved_rng_state["cuda_rng_state"] = [
                    as_cpu_byte_tensor(state)
                    for state in cuda_states
                ]

            restore_rng_state(saved_rng_state)
            print("RNG state restoration: PASSED")

        except Exception as rng_error:
            # Eski checkpoint ile yeni çalışma zamanı arasında
            # RNG biçimi uyuşmazsa kontrollü şekilde yeniden tohumla.
            seed_everything(
                int(CONFIG["experiment"]["seed"])
            )

            print(
                "RNG state biçimi farklıydı; aynı seed ile "
                "güvenli şekilde yeniden başlatıldı."
            )
            print("RNG ayrıntısı:", repr(rng_error))

    else:
        seed_everything(
            int(CONFIG["experiment"]["seed"])
        )
        print("Checkpoint RNG state yok; seed yeniden uygulandı.")

    # DataLoader generator durumunu geri yükle
    saved_loader_state = resume_state.get(
        "loader_generator_state"
    )

    if saved_loader_state is not None:
        try:
            loader_generator.set_state(
                as_cpu_byte_tensor(saved_loader_state)
            )
            print("DataLoader generator state: PASSED")

        except Exception as loader_error:
            loader_generator.manual_seed(
                int(CONFIG["experiment"]["seed"])
            )

            print(
                "DataLoader generator aynı seed ile "
                "güvenli şekilde yeniden başlatıldı."
            )
            print("DataLoader ayrıntısı:", repr(loader_error))

    else:
        loader_generator.manual_seed(
            int(CONFIG["experiment"]["seed"])
        )

    # --------------------------------------------------------
    # Eğitim geçmişini geri yükle
    # --------------------------------------------------------
    history = list(
        resume_state["history"]
    )

    best_metric_score = float(
        resume_state["best_metric_score"]
    )

    best_threshold = float(
        resume_state["best_threshold"]
    )

    best_epoch = int(
        resume_state["best_epoch"]
    )

    epochs_without_improvement = int(
        resume_state["epochs_without_improvement"]
    )

    start_epoch = int(
        resume_state["epoch"]
    ) + 1

    print()
    print(
        f"RESUME PASSED — continuing from epoch "
        f"{start_epoch}"
    )
    print("Best metric:", best_metric_score)
    print("Best threshold:", best_threshold)
    print("Device:", DEVICE)

else:
    print("NEW RUN — starting at epoch 1")

Checkpoint yükleniyor: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/20260807_2235_mouth_swinv2_tiny_seed42/checkpoints/last.ckpt
Checkpoint experiment identity: PASSED
RNG state restoration: PASSED
DataLoader generator state: PASSED

RESUME PASSED — continuing from epoch 2
Best metric: 0.7338824067719056
Best threshold: 0.37499999999999994
Device: cuda


In [24]:
# ============================================================
# 14. TRAINING LOOP
# ============================================================
import json

import numpy as np
import pandas as pd

EPOCHS = int(T["epochs"])
PATIENCE = int(T["early_stopping_patience"])
MONITOR = str(T["monitor_metric"])
FALLBACK = str(T["fallback_metric"])
AMP_OVERFLOW_ABORT_AFTER = int(
    T.get("amp_overflow_abort_after", 8)
)

if start_epoch > EPOCHS:
    print(
        "Training already reached configured epoch count. "
        "Skipping training loop."
    )

for epoch in range(start_epoch, EPOCHS + 1):
    print("\n" + "=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print("=" * 80)

    train_out = run_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        device=DEVICE,
        amp_enabled=AMP_ENABLED,
        scaler=scaler,
        gradient_clip_norm=float(
            T["gradient_clip_norm"]
        ),
        amp_overflow_abort_after=AMP_OVERFLOW_ABORT_AFTER,
        optimizer=optimizer,
    )

    train_metrics = compute_metrics(
        train_out["labels"],
        train_out["probabilities"],
        threshold=0.5,
    )

    val_out = run_epoch(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE,
        amp_enabled=AMP_ENABLED,
        scaler=scaler,
        gradient_clip_norm=float(
            T["gradient_clip_norm"]
        ),
        amp_overflow_abort_after=AMP_OVERFLOW_ABORT_AFTER,
        optimizer=None,
    )

    epoch_threshold, _ = (
        select_best_f1_threshold(
            val_out["labels"],
            val_out["probabilities"],
        )
    )

    val_metrics = compute_metrics(
        val_out["labels"],
        val_out["probabilities"],
        threshold=epoch_threshold,
    )

    score = val_metrics.get(
        MONITOR,
        float("nan"),
    )

    if not np.isfinite(score):
        score = val_metrics.get(
            FALLBACK,
            float("nan"),
        )

    if not np.isfinite(score):
        raise FloatingPointError(
            "Primary and fallback validation metric "
            "are non-finite."
        )

    row = {
        "epoch": epoch,
        "backbone_lr": optimizer.param_groups[0]["lr"],
        "head_lr": optimizer.param_groups[1]["lr"],
        "train_loss": train_out["loss"],
        "val_loss": val_out["loss"],
        "val_threshold": epoch_threshold,
        "train_amp_overflow_steps": int(
            train_out.get("amp_overflow_steps", 0)
        ),
        **{
            f"train_{key}": value
            for key, value in train_metrics.items()
        },
        **{
            f"val_{key}": value
            for key, value in val_metrics.items()
        },
    }

    history.append(row)

    improved = (
        score >
        best_metric_score + 1e-6
    )

    if improved:
        best_metric_score = float(score)
        best_threshold = float(epoch_threshold)
        best_epoch = int(epoch)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    # Advance scheduler before saving end-of-epoch state.
    scheduler.step()

    end_of_epoch_state = build_checkpoint_state(
        epoch=epoch,
        best_metric_score=best_metric_score,
        best_threshold=best_threshold,
        history=history,
        best_epoch=best_epoch,
        epochs_without_improvement=epochs_without_improvement,
    )

    # last.ckpt always represents the complete state for epoch+1 resume.
    atomic_save_checkpoint(
        end_of_epoch_state,
        LAST_CHECKPOINT,
    )

    # best.ckpt is saved from the same complete end-of-epoch state.
    if improved:
        atomic_save_checkpoint(
            end_of_epoch_state,
            BEST_CHECKPOINT,
        )
        print("New BEST checkpoint saved.")

    epoch_path = (
        CHECKPOINT_DIR /
        f"epoch_{epoch:03d}.ckpt"
    )

    atomic_save_checkpoint(
        end_of_epoch_state,
        epoch_path,
    )

    # Only after checkpoint success do we persist human-readable history.
    pd.DataFrame(history).to_csv(
        METRICS_DIR / "training_history.csv",
        index=False,
    )

    epoch_files = sorted(
        CHECKPOINT_DIR.glob(
            "epoch_*.ckpt"
        )
    )

    keep_n = int(
        T["keep_last_n_epoch_checkpoints"]
    )

    while len(epoch_files) > keep_n:
        epoch_files.pop(0).unlink()

    print(json.dumps(row, indent=2))
    print("Best score:", best_metric_score)
    print("Best epoch:", best_epoch)
    print("Threshold:", best_threshold)
    print(
        "Epochs without improvement:",
        epochs_without_improvement,
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break


Epoch 2/20


Train:   0%|          | 0/75 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/10 [00:00<?, ?it/s]

New BEST checkpoint saved.
{
  "epoch": 2,
  "backbone_lr": 9.94459753267812e-06,
  "head_lr": 9.939057285945933e-05,
  "train_loss": 0.6084210729189825,
  "val_loss": 0.5489135722856264,
  "val_threshold": 0.605,
  "train_amp_overflow_steps": 0,
  "train_accuracy": 0.6709920468815403,
  "train_precision": 0.6663934426229509,
  "train_recall": 0.6820469798657718,
  "train_f1": 0.6741293532338308,
  "train_roc_auc": 0.7310761523495539,
  "train_pr_auc": 0.7292829060305093,
  "val_accuracy": 0.7601351351351351,
  "val_precision": 0.7430555555555556,
  "val_recall": 0.7588652482269503,
  "val_f1": 0.7508771929824561,
  "val_roc_auc": 0.8183482040722947,
  "val_pr_auc": 0.8009282698989149
}
Best score: 0.8183482040722947
Best epoch: 2
Threshold: 0.605
Epochs without improvement: 0

Epoch 3/20


Train:   0%|          | 0/75 [00:00<?, ?it/s]

...

Evaluate:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "epoch": 15,
  "backbone_lr": 2.8549663646838704e-06,
  "head_lr": 2.1404630011522593e-05,
  "train_loss": 0.3387212385912728,
  "val_loss": 0.4453521548090754,
  "val_threshold": 0.38999999999999996,
  "train_amp_overflow_steps": 0,
  "train_accuracy": 0.8543323566345752,
  "train_precision": 0.858843537414966,
  "train_recall": 0.8473154362416108,
  "train_f1": 0.8530405405405406,
  "train_roc_auc": 0.930156767758322,
  "train_pr_auc": 0.9270005524955011,
  "val_accuracy": 0.8175675675675675,
  "val_precision": 0.8085106382978723,
  "val_recall": 0.8085106382978723,
  "val_f1": 0.8085106382978723,
  "val_roc_auc": 0.8813543811484786,
  "val_pr_auc": 0.875929327083185
}
Best score: 0.8858384808968199
Best epoch: 10
Threshold: 0.4049999999999999
Epochs without improvement: 5

Epoch 16/20


Train:   0%|          | 0/75 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "epoch": 16,
  "backbone_lr": 2.318019484660536e-06,
  "head_lr": 1.5498214331265904e-05,
  "train_loss": 0.32574649599866917,
  "val_loss": 0.4522759068656612,
  "val_threshold": 0.39499999999999996,
  "train_amp_overflow_steps": 0,
  "train_accuracy": 0.8627040602762662,
  "train_precision": 0.8686006825938567,
  "train_recall": 0.8540268456375839,
  "train_f1": 0.8612521150592216,
  "train_roc_auc": 0.9355568731672581,
  "train_pr_auc": 0.9377624026862874,
  "val_accuracy": 0.8175675675675675,
  "val_precision": 0.8320610687022901,
  "val_recall": 0.7730496453900709,
  "val_f1": 0.8014705882352942,
  "val_roc_auc": 0.8813086250285976,
  "val_pr_auc": 0.8759547119454242
}
Best score: 0.8858384808968199
Best epoch: 10
Threshold: 0.4049999999999999
Epochs without improvement: 6
Early stopping triggered.


In [25]:
# ============================================================
# 15. FRESH BEST.CKPT INFERENCE GATE + FINAL TEST
# ============================================================

import json
import torch

if not BEST_CHECKPOINT.is_file():
    raise RuntimeError(
        "best.ckpt was not created."
    )

best_state = load_checkpoint(
    BEST_CHECKPOINT,
    DEVICE,
)

inference_model = SwinV2TinyBinaryClassifier(
    pretrained=False,
    dropout=float(CONFIG["model"]["dropout"]),
).to(DEVICE)

inference_model.load_state_dict(
    best_state["model_state_dict"]
)

inference_model.eval()

fresh_batch = next(iter(test_loader))

with torch.no_grad():
    fresh_logits = inference_model(
        fresh_batch["image"].to(DEVICE)
    )

assert (
    fresh_logits.shape[0]
    ==
    fresh_batch["image"].shape[0]
)

assert_finite_tensor(
    fresh_logits,
    "fresh best.ckpt inference",
)

print("Fresh best.ckpt inference: PASSED")

FINAL_THRESHOLD = float(
    best_state["best_threshold"]
)

# Test is used for final evaluation only.
test_out = run_epoch(
    model=inference_model,
    loader=test_loader,
    criterion=criterion,
    device=DEVICE,
    amp_enabled=AMP_ENABLED,
    scaler=scaler,
    gradient_clip_norm=float(
        T["gradient_clip_norm"]
    ),
    amp_overflow_abort_after=int(
        T.get("amp_overflow_abort_after", 8)
    ),
    optimizer=None,
)

test_metrics = compute_metrics(
    test_out["labels"],
    test_out["probabilities"],
    threshold=FINAL_THRESHOLD,
)

test_metrics.update(
    {
        "threshold": FINAL_THRESHOLD,
        "best_epoch": int(best_state["epoch"]),
        "run_id": run_id,
    }
)

save_json_atomic(
    test_metrics,
    METRICS_DIR / "test_frame_metrics.json",
)

print(json.dumps(test_metrics, indent=2))


Fresh best.ckpt inference: PASSED


Evaluate:   0%|          | 0/10 [00:24<?, ?it/s]

{
  "accuracy": 0.804635761589404,
  "precision": 0.8169934640522876,
  "recall": 0.8012820512820513,
  "f1": 0.8090614886731392,
  "roc_auc": 0.860730593607306,
  "pr_auc": 0.8632472624649614,
  "threshold": 0.4049999999999999,
  "best_epoch": 10,
  "run_id": "20260807_2235_mouth_swinv2_tiny_seed42"
}


In [26]:
# ============================================================
# 16. FRAME-LEVEL + VIDEO-LEVEL PREDICTIONS
# ============================================================

import json
import pandas as pd

frame_predictions = (
    test_out["probabilities"]
    >= FINAL_THRESHOLD
).astype(np.int64)

predictions_df = pd.DataFrame(
    {
        "sample_id": test_out["sample_ids"],
        "video_id": test_out["video_ids"],
        "image_path": test_out["paths"],
        "true_label": test_out["labels"],
        "fake_probability": test_out["probabilities"],
        "predicted_label": frame_predictions,
        "threshold": FINAL_THRESHOLD,
    }
)

predictions_df["true_class"] = (
    predictions_df["true_label"]
    .map({0: "real", 1: "fake"})
)

predictions_df["predicted_class"] = (
    predictions_df["predicted_label"]
    .map({0: "real", 1: "fake"})
)

predictions_df.to_csv(
    PREDICTIONS_DIR / "test_frame_predictions.csv",
    index=False,
)


if video_identity_reliable:
    video_predictions_df = (
        predictions_df
        .groupby("video_id", as_index=False)
        .agg(
            true_label=("true_label", "first"),
            fake_probability=("fake_probability", "mean"),
            frame_count=("sample_id", "count"),
        )
    )

    video_predictions_df["predicted_label"] = (
        video_predictions_df["fake_probability"]
        >= FINAL_THRESHOLD
    ).astype(np.int64)

    video_metrics = compute_metrics(
        video_predictions_df["true_label"].to_numpy(),
        video_predictions_df[
            "fake_probability"
        ].to_numpy(),
        threshold=FINAL_THRESHOLD,
    )

    video_metrics.update(
        {
            "available": True,
            "threshold": FINAL_THRESHOLD,
            "run_id": run_id,
        }
    )

    video_predictions_df.to_csv(
        PREDICTIONS_DIR / "test_video_predictions.csv",
        index=False,
    )
else:
    # Do not publish misleading "video-level" metrics when metadata does not
    # contain trustworthy source-video identities.
    video_metrics = {
        "available": False,
        "run_id": run_id,
        "reason": (
            "video_id appears to be a label/split bucket rather than a true "
            "source-video identifier; per-original-video metrics are disabled."
        ),
    }

    pd.DataFrame(
        columns=[
            "video_id",
            "true_label",
            "fake_probability",
            "frame_count",
            "predicted_label",
        ]
    ).to_csv(
        PREDICTIONS_DIR / "test_video_predictions.csv",
        index=False,
    )

save_json_atomic(
    video_metrics,
    METRICS_DIR / "test_video_metrics.json",
)

print("VIDEO LEVEL")
print(json.dumps(video_metrics, indent=2))


VIDEO LEVEL
{
  "accuracy": 0.8047945205479452,
  "precision": 0.8082191780821918,
  "recall": 0.8027210884353742,
  "f1": 0.8054607508532423,
  "roc_auc": 0.859629368988975,
  "pr_auc": 0.8554669332661462,
  "available": true,
  "threshold": 0.4049999999999999,
  "run_id": "20260807_2235_mouth_swinv2_tiny_seed42"
}


In [27]:
# ============================================================
# 17. FIGURES
# ============================================================

import pandas as pd

history_df = pd.DataFrame(
    best_state.get("history", history)
)

if history_df.empty:
    raise RuntimeError(
        "Training history is empty."
    )

generate_all_figures(
    history_df=history_df,
    labels=test_out["labels"],
    probabilities=test_out["probabilities"],
    threshold=FINAL_THRESHOLD,
    figures_dir=FIGURES_DIR,
    min_short_edge=int(
        CONFIG["quality"]["min_figure_short_edge_px"]
    ),
)

print("Figures:", FIGURES_DIR)


Figures: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/20260807_2235_mouth_swinv2_tiny_seed42/figures


In [28]:
# ============================================================
# 18. FINAL QUALITY GATES + EXPERIMENT SUMMARY
# ============================================================

import json

final_summary = {
    "run_id": run_id,
    "architecture": "Swin V2 Tiny",
    "region": "Combined Mouth ROI",
    "mode": mode,
    "seed": SEED,
    "normalization": normalization_audit,
    "best_epoch": int(best_state["epoch"]),
    "best_validation_score": float(
        best_state["best_metric_score"]
    ),
    "validation_selected_threshold": FINAL_THRESHOLD,
    "test_frame_metrics": test_metrics,
    "test_video_metrics": video_metrics,
    "train_frames": len(train_df),
    "val_frames": len(val_df),
    "test_frames": len(test_df),
    "train_videos": int(
        train_df[video_col].nunique()
    ),
    "val_videos": int(
        val_df[video_col].nunique()
    ),
    "test_videos": int(
        test_df[video_col].nunique()
    ),
    "ssot_schema_missing_reported": ssot_missing,
}

save_json_atomic(
    final_summary,
    RUN_DIR / "experiment_summary.json",
)

required_outputs = [
    RUN_DIR / "environment.json",
    RUN_DIR / "requirements_runtime.lock.txt",
    RUN_DIR / "source_hash_manifest.json",
    RUN_DIR / "experiment_summary.json",
    AUDIT_DIR / "metadata_schema_audit.json",
    AUDIT_DIR / "normalization_audit.json",
    AUDIT_DIR / "video_identity_audit.json",
    AUDIT_DIR / "input_numerical_sanity.json",
    AUDIT_DIR / "dataset_summary.csv",
    AUDIT_DIR / "validated_training_manifest.csv",
    AUDIT_DIR / "split_leakage_report.json",
    AUDIT_DIR / "checkpoint_continuity_test.json",
    CHECKPOINT_DIR / "last.ckpt",
    CHECKPOINT_DIR / "best.ckpt",
    METRICS_DIR / "training_history.csv",
    METRICS_DIR / "test_frame_metrics.json",
    METRICS_DIR / "test_video_metrics.json",
    PREDICTIONS_DIR / "test_frame_predictions.csv",
    PREDICTIONS_DIR / "test_video_predictions.csv",
    FIGURES_DIR / "training_validation_loss.png",
    FIGURES_DIR / "training_validation_accuracy.png",
    FIGURES_DIR / "validation_roc_auc.png",
    FIGURES_DIR / "test_confusion_matrix.png",
    FIGURES_DIR / "test_roc_curve.png",
    FIGURES_DIR / "test_precision_recall_curve.png",
]

missing = [
    str(path)
    for path in required_outputs
    if not path.exists()
]

if missing:
    raise RuntimeError(
        "Final quality gate FAILED. Missing:\n"
        + "\n".join(missing)
    )

# Final leakage re-check.
assert train_videos.isdisjoint(val_videos)
assert train_videos.isdisjoint(test_videos)
assert val_videos.isdisjoint(test_videos)

# Figure resolution re-check.
from PIL import Image

for figure_path in FIGURES_DIR.glob("*.png"):
    with Image.open(figure_path) as image:
        assert (
            min(image.size)
            >= int(
                CONFIG["quality"][
                    "min_figure_short_edge_px"
                ]
            )
        ), (
            f"Figure quality failed: "
            f"{figure_path} -> {image.size}"
        )

# Best checkpoint can be loaded from scratch.
final_check = load_checkpoint(
    BEST_CHECKPOINT,
    DEVICE,
)

assert (
    int(final_check["epoch"])
    == int(best_state["epoch"])
)

print("=" * 80)
print("ALL FINAL QUALITY GATES PASSED")
print("=" * 80)
print("RUN DIRECTORY:")
print(RUN_DIR)
print("\nFRAME LEVEL:")
print(json.dumps(test_metrics, indent=2))
print("\nVIDEO LEVEL:")
print(json.dumps(video_metrics, indent=2))


ALL FINAL QUALITY GATES PASSED
RUN DIRECTORY:
/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/20260807_2235_mouth_swinv2_tiny_seed42

FRAME LEVEL:
{
  "accuracy": 0.804635761589404,
  "precision": 0.8169934640522876,
  "recall": 0.8012820512820513,
  "f1": 0.8090614886731392,
  "roc_auc": 0.860730593607306,
  "pr_auc": 0.8632472624649614,
  "threshold": 0.4049999999999999,
  "best_epoch": 10,
  "run_id": "20260807_2235_mouth_swinv2_tiny_seed42"
}

VIDEO LEVEL:
{
  "accuracy": 0.8047945205479452,
  "precision": 0.8082191780821918,
  "recall": 0.8027210884353742,
  "f1": 0.8054607508532423,
  "roc_auc": 0.859629368988975,
  "pr_auc": 0.8554669332661462,
  "available": true,
  "threshold": 0.4049999999999999,
  "run_id": "20260807_2235_mouth_swinv2_tiny_seed42"
}


## Resume nasıl kullanılır?

Colab veya GPU oturumu kesilirse **aynı run klasöründen devam etmek** için Config hücresinde yalnızca:

```python
"mode": "resume",
"resume_run_id": "20260807_...._mouth_swinv2_tiny_seed42",
```

olarak değiştirin ve notebook'u baştan çalıştırın.

Resume aşaması `last.ckpt` içinden:
- model,
- optimizer,
- scheduler,
- AMP scaler,
- Python / NumPy / Torch / CUDA RNG,
- DataLoader generator state,
- history,
- best metric,
- best epoch,
- early-stopping counter,
- validation threshold

bilgilerini geri yükler ve `epoch + 1`'den devam eder.